# Product Evidence Platform — Final Manufacturer-First Azure ML Notebook

Single supported notebook.

```text
offline product interpretation
→ manufacturer_primary
→ requested_retailer_country / country_alternative
→ global_fallback
→ exact identity + browser + requested-feature + scrapability + durability gates
→ manufacturer-first authority ranking
→ primary_url + manufacturer_url + retailer_url
```

A qualified official manufacturer page is the product-truth source. Retailer fallback is used when manufacturer evidence is missing, inaccessible, incomplete, non-product, transient, or wrong-product/variant.


In [ ]:
from __future__ import annotations
import importlib, json, subprocess, sys
from pathlib import Path
from pprint import pprint

AUTO_RECOVER_PLATFORM = True
CLEAN_BUILD_ON_RECOVERY = True

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "src" / "product_evidence_harness").is_dir():
            return candidate
    raise RuntimeError("Could not locate the web_search_tool repository root")

PROJECT_ROOT = find_project_root()
SRC_PATH = (PROJECT_ROOT / "src").resolve()
LOCAL_PACKAGE = (SRC_PATH / "product_evidence_harness").resolve()
if not (LOCAL_PACKAGE / "notebook_runtime.py").is_file():
    raise FileNotFoundError("notebook_runtime.py missing. Pull master and restart the kernel.")

for required_path in (str(PROJECT_ROOT), str(SRC_PATH)):
    sys.path[:] = [item for item in sys.path if str(Path(item or ".").resolve()) != required_path]
    sys.path.insert(0, required_path)
for module_name in tuple(sys.modules):
    if module_name == "product_evidence_harness" or module_name.startswith("product_evidence_harness.") or module_name == "src.product_evidence_harness" or module_name.startswith("src.product_evidence_harness."):
        del sys.modules[module_name]
importlib.invalidate_caches()

PACKAGE_IMPORTS = {"pandas":"pandas>=2.2,<3","matplotlib":"matplotlib>=3.8,<4","seaborn":"seaborn>=0.13.2,<1","rich":"rich>=13.7,<15","openpyxl":"openpyxl>=3.1,<4"}
missing_specs = []
for module_name, package_spec in PACKAGE_IMPORTS.items():
    try: importlib.import_module(module_name)
    except ImportError: missing_specs.append(package_spec)
if missing_specs: subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_specs])

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from rich.console import Console
import product_evidence_harness
loaded_package = Path(product_evidence_harness.__file__).resolve()
if LOCAL_PACKAGE not in loaded_package.parents:
    raise RuntimeError(f"Wrong package loaded. Expected {LOCAL_PACKAGE}; loaded {loaded_package}")

from product_evidence_harness.notebook_runtime import AGENT_URL, DEFAULT_FEATURE_SET, HEARTBEAT_SECONDS, ensure_platform_ready, host_artifact_dir, run_product
from product_evidence_harness.notebook_diagnostics import build_single_product_diagnostics, display_compact, display_rich_summary, plot_all_diagnostics, plot_candidate_outcomes, plot_confidence_distribution, plot_confidence_vs_coverage, plot_domain_quality, plot_feature_heatmap, plot_funnel, plot_rejection_reasons, plot_stage_yield
from product_evidence_harness.source_authority_notebook import apply_source_authority_notebook_patch
from product_evidence_harness.adaptive_notebook_diagnostics import build_adaptive_search_diagnostics, display_adaptive_search_summary, export_adaptive_search_tables, plot_credit_progression, plot_engine_candidate_yield, plot_engine_credit_allocation

apply_source_authority_notebook_patch()
sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 200)

health, platform_recovery = ensure_platform_ready(PROJECT_ROOT, auto_recover=AUTO_RECOVER_PLATFORM, clean_build=CLEAN_BUILD_ON_RECOVERY)
feature_sets = sorted(path.stem for path in (PROJECT_ROOT / "inputs" / "private").glob("*.json"))
if DEFAULT_FEATURE_SET not in feature_sets: raise RuntimeError("The committed inputs/private/toy_features.json is missing")
if not health.get("manufacturer_first_primary_url"): raise RuntimeError("Manufacturer-first capability missing; run a clean build.")

platform_readiness_df = pd.DataFrame([{
    "repository":str(PROJECT_ROOT),"package":str(loaded_package),"agent":AGENT_URL,
    "runtime_contract":health.get("runtime_contract_version"),"platform_status":health.get("status"),
    "manufacturer_first_primary_url":health.get("manufacturer_first_primary_url"),
    "auto_recovery_attempted":platform_recovery.attempted,"auto_recovery_succeeded":platform_recovery.recovered,
    "clean_build_used":platform_recovery.clean_build,"recovery_trigger":platform_recovery.trigger,
    "recovery_elapsed_seconds":platform_recovery.elapsed_seconds,"default_feature_set":DEFAULT_FEATURE_SET,
    "available_feature_sets":", ".join(feature_sets),
}])
Console().print(f"Runtime: {health.get('runtime_contract_version')} | manufacturer_first_primary_url={health.get('manufacturer_first_primary_url')}")
display(platform_readiness_df)
pprint(health)


## 1. Run one product

`main_text` and `country_code` are mandatory. Keep EAN/GTIN as text. Set `RUN_SINGLE_PRODUCT = True` only after replacing the sample. `REVIEW_REQUIRED` delivers a real review URL; no safe direct page raises `MANDATORY_PRODUCT_URL_NOT_FOUND`.


In [ ]:
FEATURE_SET = 'toy_features'
RUN_SINGLE_PRODUCT = False
product = {'row_id':'ROW-001','main_text':'Replace with the vendor product main text','country_code':'CZ','retailer_name':None,'ean':None,'language_code':None}

if RUN_SINGLE_PRODUCT:
    if product['main_text'].startswith('Replace with'): raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    search = result.get('search') or {}
    acceptance = result.get('primary_url_acceptance') or {}
    delivery = result.get('url_delivery') or {}
    source_selection = result.get('source_selection') or {}
    pprint({
        'row_id':(result.get('product') or {}).get('row_id'),'job_status':result.get('job_status'),
        'primary_url':result.get('primary_url'),'primary_url_role':result.get('primary_url_role'),
        'manufacturer_url':result.get('manufacturer_url'),'retailer_url':result.get('retailer_url'),
        'source_selection_reason':source_selection.get('selection_reason'),
        'url_delivery_status':delivery.get('status'),'strictly_verified':delivery.get('strictly_verified'),
        'strict_primary_accepted':acceptance.get('accepted'),'delivered':delivery.get('delivered'),
        'engine_sequence':search.get('engine_sequence'),'target_source_tiers':search.get('target_source_tiers'),
        'market_decision_path':search.get('market_decision_path'),'serpapi_requests_used':search.get('serpapi_requests_used'),
        'search_stop_reason':search.get('stop_reason'),'product_identification':result.get('product_identification'),
    })
else:
    print('Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.')


## 2. Manufacturer-first decision, belief state, and complete diagnostics


In [ ]:
if 'result' not in globals(): raise RuntimeError('Run the single-product cell first')
artifact_path = host_artifact_dir(PROJECT_ROOT,result)
if artifact_path is None or not artifact_path.is_dir(): raise RuntimeError('Repository-local artifact directory was not found')

belief = json.loads((artifact_path / 'product_belief.json').read_text(encoding='utf-8'))
leading = belief.get('leading_hypothesis') or {}
metrics = belief.get('metrics') or {}
product_identification_df = pd.DataFrame([{'identified_product':leading.get('canonical_name'),'resolution_status':belief.get('resolution_status'),'leading_probability':leading.get('posterior_probability'),'posterior_margin':metrics.get('posterior_margin'),'search_readiness':metrics.get('search_readiness'),'identity_completeness':metrics.get('identity_completeness'),'ambiguity_entropy':metrics.get('ambiguity_entropy'),'assumption_burden':metrics.get('assumption_burden')}])
hypotheses_df = pd.DataFrame(belief.get('hypotheses') or [])
uncertainties_df = pd.DataFrame(belief.get('uncertainties') or [])
belief_updates_df = pd.DataFrame(belief.get('snapshots') or [])
evidence_ledger_df = pd.DataFrame(belief.get('evidence_ledger') or [])

source_selection = result.get('source_selection') or {}
source_selection_df = pd.DataFrame([{'policy':source_selection.get('policy'),'primary_url':result.get('primary_url'),'primary_url_role':result.get('primary_url_role'),'manufacturer_url':result.get('manufacturer_url'),'retailer_url':result.get('retailer_url'),'selected_source_tier_name':source_selection.get('selected_source_tier_name'),'selected_source_role':source_selection.get('selected_source_role'),'selection_reason':source_selection.get('selection_reason')}])
source_selection_path = artifact_path / 'source_selection.json'
if not source_selection_path.is_file(): raise RuntimeError(f'Missing manufacturer-first decision artifact: {source_selection_path}')

diagnostics = build_single_product_diagnostics(result, artifact_dir=artifact_path)
adaptive_diagnostics = build_adaptive_search_diagnostics(result)
overview_df=diagnostics.overview_df; search_stages_df=diagnostics.search_stages_df; serp_results_df=diagnostics.serp_results_df; results_df=diagnostics.results_df
agentic_df=diagnostics.agentic_df; feature_evidence_df=diagnostics.feature_evidence_df; feature_matrix_df=diagnostics.feature_matrix_df
funnel_df=diagnostics.funnel_df; domain_summary_df=diagnostics.domain_summary_df; stage_quality_df=diagnostics.stage_quality_df
rejection_reasons_df=diagnostics.rejection_reasons_df; selection_rca_df=diagnostics.selection_rca_df
search_actions_df=adaptive_diagnostics.search_actions_df; search_engine_summary_df=adaptive_diagnostics.search_engine_summary_df
search_handles_df=adaptive_diagnostics.search_handles_df; search_decision_rca_df=adaptive_diagnostics.search_decision_rca_df
source_hierarchy_df=search_actions_df[[c for c in ('serp_credit','target_source_tier','market_stage','engine','purpose','planner_source','results_returned','new_candidate_urls','candidates_qualified','candidates_scraped','working_url_found','reason','query') if c in search_actions_df]].copy()
source_tier_summary_df=(results_df.groupby(['source_tier','source_tier_name','source_role'],dropna=False).agg(candidate_urls=('url','count'),scrape_attempted=('scrape_attempted','sum'),scrape_accepted=('scrape_success','sum'),identity_accepted=('identity_accepted','sum'),selected_urls=('strict_selected','sum'),mean_confidence=('confidence','mean')).reset_index() if not results_df.empty and 'source_tier' in results_df else pd.DataFrame())
url_delivery_df=pd.DataFrame([{**(result.get('url_delivery') or {}),'job_status':result.get('job_status'),'strict_acceptance':(result.get('primary_url_acceptance') or {}).get('accepted'),'primary_url':result.get('primary_url'),'primary_url_role':result.get('primary_url_role'),'manufacturer_url':result.get('manufacturer_url'),'retailer_url':result.get('retailer_url')}])

display_rich_summary(diagnostics); display_adaptive_search_summary(adaptive_diagnostics)
for frame,title,rows in [
    (source_selection_df,'Manufacturer-first source selection',5),(url_delivery_df,'Mandatory product URL delivery',5),
    (source_hierarchy_df,'Source hierarchy by SerpAPI credit',10),(search_actions_df,'Adaptive SerpAPI decisions',10),
    (search_engine_summary_df,'Search-engine yield and conversion',20),(search_decision_rca_df,'Search decision RCA',40),
    (search_handles_df,'Product tokens, identifiers and image handles',30),(source_tier_summary_df,'Candidate conversion by source tier',20),
    (results_df,'One canonical product URL candidate per row',100),(selection_rca_df,'Final URL selection RCA',30),
    (funnel_df,'Candidate conversion funnel',20),(rejection_reasons_df,'Rejection and review reasons',40),
    (agentic_df,'Agentic-browser investigations',30),(feature_evidence_df,'URL-feature evidence',100),
]:
    display_compact(frame,title=title,max_rows=rows)
if not results_df.empty: assert results_df['url'].is_unique

plot_engine_credit_allocation(adaptive_diagnostics); plot_engine_candidate_yield(adaptive_diagnostics); plot_credit_progression(adaptive_diagnostics)
plot_funnel(diagnostics); plot_stage_yield(diagnostics); plot_candidate_outcomes(diagnostics); plot_confidence_distribution(diagnostics)
plot_confidence_vs_coverage(diagnostics); plot_domain_quality(diagnostics); plot_rejection_reasons(diagnostics); plot_feature_heatmap(diagnostics); plot_all_diagnostics(diagnostics)


## 3. Export the complete review workbook


In [ ]:
tables=diagnostics.tables()
tables.update({'platform_readiness':platform_readiness_df,'product_identification':product_identification_df,'product_hypotheses':hypotheses_df,'product_uncertainties':uncertainties_df,'belief_updates':belief_updates_df,'evidence_ledger':evidence_ledger_df,'source_selection':source_selection_df,'url_delivery':url_delivery_df,'source_hierarchy':source_hierarchy_df,'source_tier_summary':source_tier_summary_df})
workbook_path=artifact_path / 'single_product_diagnostics.xlsx'
with pd.ExcelWriter(workbook_path,engine='openpyxl') as writer:
    for sheet_name,frame in tables.items():
        if isinstance(frame,pd.DataFrame): frame.to_excel(writer,sheet_name=str(sheet_name)[:31],index=False)
export_adaptive_search_tables(adaptive_diagnostics,workbook_path)
print(f'Workbook: {workbook_path}')
print(f'Adaptive trace: {artifact_path / "adaptive_search_trace.json"}')
print(f'Mandatory delivery: {artifact_path / "mandatory_url_delivery.json"}')
print(f'Source selection: {artifact_path / "source_selection.json"}')


## Reviewer checklist

Validate exact identity, product form, model, variant, size, quantity, pack, and requested features on `primary_url`. Use `retailer_url` for price, stock, local market, language, and purchase context. Confirm manufacturer primary only after all mandatory gates; confirm retailer fallback when manufacturer evidence is inadequate. `REVIEW_REQUIRED` is a delivered review candidate, not an automated exact-match claim.
